# 08 · Grad-CAM: Where Does the Model Look?

**Question:** notebook 06 showed that attention does **not** improve accuracy (the baseline is already near 99%).
Does attention instead make the model **look at the ulcer more precisely**?

**What Grad-CAM is:** a heatmap showing which parts of the image pushed the model towards "ulcer".
Red = strong evidence, blue = ignored.

**How we measure it (not just pictures):**
The Kaggle DFU dataset has no ulcer outlines, so we use **FUSeg** (MICCAI 2021 Foot Ulcer Segmentation Challenge, Wang et al. 2020, *Sci Rep*).
It is public and has **expert-drawn ulcer masks**. None of these images were used in training, so every image is new to every model.

| Measure | Meaning | Higher = better |
|---|---|---|
| **Energy inside ulcer** (primary) | Share of the heatmap that falls on the ulcer | ✓ |
| **Pointing game** | Is the hottest pixel on (or within 15 px of) the ulcer? | ✓ |
| **IoU** | Overlap between the hot area (heatmap ≥ 0.5) and the ulcer | ✓ |

**Two reference lines**, so the numbers can be judged:
- **Chance:** a flat heatmap. Its energy inside the ulcer equals the ulcer's share of the image.
- **Centre guess:** a blob in the image centre, with no model at all. A model must beat this to show real localisation.

**Fixed before seeing results:** the primary measure is energy inside the ulcer, and the primary comparison is the
**attention variant selected in notebook 06 vs the Baseline** (paired Wilcoxon test over images). The other variants are exploratory.

**Why crops:** FUSeg photos show the whole foot, and the ulcer is often below 1% of the image. Our models were trained on close-up patches,
and Grad-CAM works on a coarse 7×7 grid. So each image is cropped to a square window around the ulcer, and the window is **shifted randomly**
so the ulcer is not always in the centre. The centre-guess line checks that this shift is enough.

**Inputs (Add Input):** the dataset **laithjj/diabetic-foot-ulcer-dfu**, **DFU 03 - Leakage-Free Split** and **DFU 06 - LiteNet Ablation**.
**Accelerator:** GPU T4 (CPU also works, slower) · **Internet: On** (to download FUSeg) · No training, about 10-20 minutes.

In [ ]:
import json, os, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers, ops
from PIL import Image
from scipy.ndimage import distance_transform_edt
from scipy.stats import wilcoxon

CONFIG = {
    "VARIANTS": ["Baseline", "ECA-Late", "CA-Late", "ECA-Multi", "CA-Multi"],
    "FOLDS": [0, 1, 2, 3, 4],
    "IMG_SIZE": 224,
    "FUSEG_SPLITS": ["train", "validation"],   # both have expert masks; no image was used in our training
    "CROP_SCALE": 2.5,        # crop side = 2.5 x the ulcer's longest side
    "MIN_CROP": 96,           # pixels, in the original 512x512 image
    "JITTER": 0.5,            # random shift: up to half the free space on each side (ulcer stays fully inside)
    "POINTING_TOL": 15,       # pixels at 224x224 (as in the original pointing game)
    "CAM_THRESHOLD": 0.5,     # hot area for IoU
    "N_BOOT": 2000,
    "SEED": 42,
}
S = CONFIG["IMG_SIZE"]
INPUT = Path("/kaggle/input")

PATCHES = next(INPUT.rglob("Patches"))                        # Kaggle DFU dataset
SPLITS = next(INPUT.rglob("splits.csv"))                      # notebook 03
WEIGHTS = next(INPUT.rglob("Baseline_fold0.npz")).parent      # notebook 06
OUT06 = WEIGHTS.parent
SELECTED = json.load(open(OUT06 / "selection.json"))["selected_attention"]
print("Weights from:", WEIGHTS)
print("Attention variant selected in notebook 06:", SELECTED)

## 1. Download FUSeg (images + expert ulcer masks)
Only the two labelled folders are downloaded from the official challenge repository (github.com/uwm-bigdata/wound-segmentation).

In [ ]:
FUSEG = Path("/tmp/fuseg_repo")
SUB = "data/Foot Ulcer Segmentation Challenge"
if not (FUSEG / SUB / "validation").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/uwm-bigdata/wound-segmentation.git", str(FUSEG)], check=True)
    subprocess.run(["git", "-C", str(FUSEG), "sparse-checkout", "set", "--no-cone"]
                   + [f"/{SUB}/{s}/" for s in CONFIG["FUSEG_SPLITS"]], check=True)
for s in CONFIG["FUSEG_SPLITS"]:
    print(s, "images:", len(list((FUSEG / SUB / s / "images").glob("*.png"))),
          "| masks:", len(list((FUSEG / SUB / s / "labels").glob("*.png"))))

## 2. Build the ulcer crops
Each crop contains the whole ulcer, shifted randomly (fixed seed). Images with an empty mask are skipped and counted.

In [ ]:
rng = np.random.default_rng(CONFIG["SEED"])

def crop_box(m):
    H, W = m.shape
    ys, xs = np.where(m)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    h, w = y1 - y0, x1 - x0
    side = int(min(max(max(h, w) * CONFIG["CROP_SCALE"], CONFIG["MIN_CROP"], h, w), H, W))
    def start(lo, hi, size, limit):
        slack = (side - (hi - lo)) / 2
        centre = (lo + hi) / 2 + rng.uniform(-1, 1) * CONFIG["JITTER"] * slack
        return int(np.clip(round(centre - side / 2), 0, limit - side))
    return start(y0, y1, h, H), start(x0, x1, w, W), side

imgs, masks, meta = [], [], []
skipped = 0
for s in CONFIG["FUSEG_SPLITS"]:
    for f in sorted((FUSEG / SUB / s / "labels").glob("*.png")):
        img_path = FUSEG / SUB / s / "images" / f.name
        if not img_path.exists():
            skipped += 1; continue
        m = np.asarray(Image.open(f).convert("L")) > 127
        if m.sum() == 0:
            skipped += 1; continue
        im = Image.open(img_path).convert("RGB")
        top, left, side = crop_box(m)
        box = (left, top, left + side, top + side)
        imgs.append(np.asarray(im.crop(box).resize((S, S), Image.BILINEAR), dtype=np.uint8))
        mc = Image.fromarray(m[top:top + side, left:left + side].astype(np.uint8) * 255).resize((S, S), Image.BILINEAR)
        mc = np.asarray(mc) > 127
        masks.append(mc)
        meta.append({"split": s, "file": f.name, "crop_side": side, "ulcer_area_in_crop": mc.mean(),
                     "ulcer_area_in_photo": m.mean()})

X_fu = np.stack(imgs)                       # uint8, (N, 224, 224, 3)
M_fu = np.stack(masks)                      # bool,  (N, 224, 224)
meta = pd.DataFrame(meta)
keep = M_fu.reshape(len(M_fu), -1).any(1)   # a tiny ulcer can vanish when resized
X_fu, M_fu, meta = X_fu[keep], M_fu[keep], meta[keep].reset_index(drop=True)
D_fu = np.stack([distance_transform_edt(~m) <= CONFIG["POINTING_TOL"] for m in M_fu])   # mask + 15 px margin
print(f"FUSeg crops: {len(X_fu)} | skipped: {skipped + int((~keep).sum())}")
print("Ulcer share of the original photo (median): {:.2%}".format(meta.ulcer_area_in_photo.median()))
print("Ulcer share of the crop (median):           {:.2%}".format(meta.ulcer_area_in_crop.median()))

fig, ax = plt.subplots(2, 6, figsize=(15, 5.4))
for j, i in enumerate(rng.choice(len(X_fu), 6, replace=False)):
    ax[0, j].imshow(X_fu[i]); ax[0, j].set_title(meta.file[i], fontsize=8)
    ax[1, j].imshow(X_fu[i]); ax[1, j].contour(M_fu[i], levels=[0.5], colors="lime", linewidths=1.5)
for a in ax.ravel(): a.axis("off")
ax[0, 0].set_ylabel("crop"); plt.suptitle("FUSeg crops (bottom: expert ulcer outline in green)"); plt.tight_layout(); plt.show()

## 3. Rebuild the notebook 06 models and load their weights
This is the **same model code as notebook 06**. Each of the 25 trained models (5 variants × 5 folds) is loaded from its saved weights.

**Grad-CAM for this head:** our models end with global average pooling and one output neuron. For this design Grad-CAM is mathematically
identical to the classic CAM: the heatmap is the last feature map weighted by the output neuron's weights (Selvaraju et al., 2017, Section 3).
We compute it for the **logit** (the score before the sigmoid), because the sigmoid saturates near 0.99-1.0 and would shrink the gradients.
Section 4 checks this equivalence numerically.

In [ ]:
class ECA(layers.Layer):
    def __init__(self, gamma=2, b=1, **kw):
        super().__init__(**kw); self.gamma, self.b = gamma, b
    def build(self, shape):
        t = int(abs((np.log2(shape[-1]) + self.b) / self.gamma))
        k = t if t % 2 else t + 1
        self.conv = layers.Conv1D(1, k, padding="same", use_bias=False, kernel_initializer="zeros")
        self.conv.build((None, shape[-1], 1))
    def call(self, x):
        z = ops.expand_dims(ops.mean(x, axis=(1, 2)), -1)
        gate = 2.0 * ops.sigmoid(self.conv(z))
        return x * ops.reshape(gate, (-1, 1, 1, x.shape[-1]))

class CoordAtt(layers.Layer):
    def __init__(self, reduction=32, **kw):
        super().__init__(**kw); self.reduction = reduction
    def build(self, shape):
        c = shape[-1]; mip = max(8, c // self.reduction)
        self.conv1 = layers.Conv2D(mip, 1, use_bias=False)
        self.bn = layers.BatchNormalization(momentum=0.9)
        self.conv_h = layers.Conv2D(c, 1, kernel_initializer="zeros")
        self.conv_w = layers.Conv2D(c, 1, kernel_initializer="zeros")
        self.conv1.build((None, None, 1, c)); self.bn.build((None, None, 1, mip))
        self.conv_h.build((None, None, 1, mip)); self.conv_w.build((None, 1, None, mip))
    def call(self, x, training=None):
        h, w = x.shape[1], x.shape[2]
        x_h = ops.mean(x, axis=2, keepdims=True)
        x_w = ops.transpose(ops.mean(x, axis=1, keepdims=True), (0, 2, 1, 3))
        y_ = ops.relu(self.bn(self.conv1(ops.concatenate([x_h, x_w], axis=1)), training=training))
        a_h = 2.0 * ops.sigmoid(self.conv_h(y_[:, :h]))
        a_w = 2.0 * ops.sigmoid(self.conv_w(ops.transpose(y_[:, h:], (0, 2, 1, 3))))
        return x * a_h * a_w

ATT = {"ECA": ECA, "CA": CoordAtt}
STAGE_ENDS = ["expanded_conv_2_add", "expanded_conv_7_add"]

def build_model(variant):
    # weights=None: the architecture is identical; all weights are then loaded from notebook 06
    base = keras.applications.MobileNetV3Small(include_top=False, weights=None,
                                               input_shape=(S, S, 3), include_preprocessing=True)
    kind, _, where = variant.partition("-")
    net = base
    if where == "Multi":
        def call_fn(layer, *args, **kwargs):
            out = layer(*args, **kwargs)
            return ATT[kind](name=f"att_{layer.name}")(out) if layer.name in STAGE_ENDS else out
        net = keras.models.clone_model(base, clone_function=lambda l: l, call_function=call_fn)
    x = net.output
    if where:
        x = ATT[kind](name="att_final")(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="classifier")(x)
    return keras.Model(net.input, out, name=variant.replace("-", "_"))

def load_trained(variant, fold):
    model = build_model(variant)
    f = np.load(WEIGHTS / f"{variant}_fold{fold}.npz")
    n = len(model.get_weights())
    assert len(f.files) - 1 == n, f"{variant} fold {fold}: {len(f.files) - 1} saved arrays, model has {n}"
    model.set_weights([f[f"w{i}"] for i in range(n)])
    feat = keras.Model(model.input, [model.get_layer("gap").input, model.output])   # feature map entering the pooling
    W, b = model.get_layer("classifier").get_weights()
    return model, feat, W[:, 0], float(b[0]), float(f["threshold"])

def gradcam(feat, w, b, X, batch=64):
    # returns 224x224 heatmaps in [0, 1], probabilities, and the logit rebuilt from the heatmap (a correctness check)
    A, p = feat.predict(X.astype(np.float32), batch_size=batch, verbose=0)
    raw = np.einsum("bhwc,c->bhw", A, w)                       # weighted feature map = Grad-CAM for the logit
    logit = raw.mean(axis=(1, 2)) + b
    cam = tf.image.resize(np.maximum(raw, 0)[..., None], (S, S), method="bilinear").numpy()[..., 0]
    mx = cam.max(axis=(1, 2), keepdims=True)
    cam = np.where(mx > 0, cam / np.maximum(mx, 1e-12), 0.0)
    return cam, p.ravel(), logit

## 4. Correctness checks
1. **Weights loaded correctly:** each model re-predicts its own test fold of the Kaggle DFU data, and the probabilities must match notebook 06.
2. **The logit rebuilt from the heatmap** must match the model's actual output.
3. **Gradient check:** the classic gradient-based Grad-CAM, computed with automatic differentiation, must give the same heatmap.

In [ ]:
splits = pd.read_csv(SPLITS)
kd = splits[splits.keep].reset_index(drop=True)
X_kd = np.stack([np.asarray(Image.open(PATCHES / r["class"] / r.file).convert("RGB").resize((S, S), Image.BILINEAR), dtype=np.uint8)
                 for _, r in kd.iterrows()])
pred06 = pd.read_csv(OUT06 / "test_predictions_all.csv")

# check 3 (gradient-based Grad-CAM) on one model; gradients of the probability, so saturated images are left out
model, feat, w, b, thr = load_trained(SELECTED, 0)
head_in = keras.Input(feat.output[0].shape[1:])
head = keras.Model(head_in, model.get_layer("classifier")(model.get_layer("dropout")(model.get_layer("gap")(head_in))))
xb = tf.constant(X_fu[:32].astype(np.float32))
A = feat(xb)[0]
with tf.GradientTape() as tape:
    tape.watch(A)
    prob = head(A, training=False)
g = tape.gradient(prob, A).numpy()
gc = np.maximum(np.einsum("bhwc,bc->bhw", A.numpy(), g.mean(axis=(1, 2))), 0)
cm = np.maximum(np.einsum("bhwc,c->bhw", A.numpy(), w), 0)
pp = prob.numpy().ravel(); ok = (pp * (1 - pp) > 1e-6) & (cm.reshape(32, -1).max(1) > 0)
corr = [np.corrcoef(gc[i].ravel(), cm[i].ravel())[0, 1] for i in np.where(ok)[0]]
print(f"Gradient Grad-CAM vs our heatmap: min correlation {min(corr):.6f} over {len(corr)} images (should be 1.000000)")
keras.backend.clear_session()

## 5. Heatmaps for all 25 models
For every model we compute the heatmap on all FUSeg crops and score it against the expert mask.
The checks from section 4 (1 and 2) run inside this loop, one line per model.

In [ ]:
def scores(cam, mask, dil):
    tot = cam.sum(axis=(1, 2))
    energy = np.where(tot > 0, (cam * mask).sum(axis=(1, 2)) / np.maximum(tot, 1e-12), np.nan)
    peak = cam.reshape(len(cam), -1).argmax(1)
    hit = np.where(tot > 0, dil.reshape(len(dil), -1)[np.arange(len(dil)), peak], np.nan)
    hot = cam >= CONFIG["CAM_THRESHOLD"]
    iou = (hot & mask).sum(axis=(1, 2)) / np.maximum((hot | mask).sum(axis=(1, 2)), 1)
    return energy, hit.astype(float), iou

EXAMPLES = np.random.default_rng(CONFIG["SEED"] + 1).choice(len(X_fu), 6, replace=False)   # fixed before results
ex_cam = {v: np.zeros((len(EXAMPLES), S, S)) for v in CONFIG["VARIANTS"]}
KD_EX = np.random.default_rng(CONFIG["SEED"] + 2).choice(np.where(kd.label == 1)[0], 6, replace=False)
kd_cam = {v: np.zeros((len(KD_EX), S, S)) for v in CONFIG["VARIANTS"]}

rows, checks = [], []
t0 = time.time()
for v in CONFIG["VARIANTS"]:
    for k in CONFIG["FOLDS"]:
        model, feat, w, b, thr = load_trained(v, k)
        # check 1: re-predict this fold's own test images and compare with notebook 06
        te = np.where(kd.fold == k)[0]
        cam_kd, p_kd, logit_kd = gradcam(feat, w, b, X_kd[te])
        ref = pred06[(pred06.model == v) & (pred06.fold == k)].set_index(["class", "file"]).prob_ulcer
        diff = np.abs(p_kd - ref.loc[list(zip(kd["class"][te], kd.file[te]))].values).max()
        # check 2: logit rebuilt from the heatmap
        logit_err = np.abs(1 / (1 + np.exp(-logit_kd)) - p_kd).max()
        for j, i in enumerate(KD_EX):            # each example image uses the model for which it was a test image
            if kd.fold[i] == k:
                kd_cam[v][j] = cam_kd[list(te).index(i)]
        # FUSeg
        cam, p, _ = gradcam(feat, w, b, X_fu)
        e, h, u = scores(cam, M_fu, D_fu)
        rows.append(pd.DataFrame({"variant": v, "fold": k, "img": np.arange(len(X_fu)), "energy": e, "hit": h, "iou": u,
                                  "prob_ulcer": p, "pred_ulcer": p >= thr}))
        ex_cam[v] += cam[EXAMPLES] / len(CONFIG["FOLDS"])
        checks.append({"variant": v, "fold": k, "max_prob_diff_vs_nb06": diff, "max_logit_check_err": logit_err})
        print(f"{v:10s} fold {k} | prob diff vs nb06 {diff:.1e} | logit check {logit_err:.1e} | "
              f"energy {np.nanmean(e):.3f} | {time.time() - t0:.0f}s", flush=True)
        keras.backend.clear_session()

per_fold = pd.concat(rows, ignore_index=True)
checks = pd.DataFrame(checks)
assert checks.max_prob_diff_vs_nb06.max() < 1e-2, "weights did not reproduce notebook 06 - stop and check the inputs"
assert checks.max_logit_check_err.max() < 1e-3, "heatmap does not reproduce the model output"
print("\nAll checks passed. Largest probability difference vs notebook 06:", f"{checks.max_prob_diff_vs_nb06.max():.1e}")
per_fold.to_csv("fuseg_scores_per_fold.csv", index=False)

## 6. Results: does attention look at the ulcer more precisely?
Each image's score is averaged over the 5 fold models. The test is paired (same images), Wilcoxon signed-rank.
The 95% interval for the difference comes from bootstrapping images.

In [ ]:
per_img = per_fold.groupby(["variant", "img"])[["energy", "hit", "iou", "pred_ulcer"]].mean().reset_index()
per_img.to_csv("fuseg_scores_per_image.csv", index=False)

# reference lines: a flat heatmap (chance) and a centre blob (no model)
yy, xx = np.mgrid[:S, :S]
centre = np.exp(-((yy - S / 2) ** 2 + (xx - S / 2) ** 2) / (2 * (S / 4) ** 2))
ce, ch, cu = scores(np.repeat(centre[None], len(M_fu), 0), M_fu, D_fu)
refs = {"Chance (flat heatmap)": {"energy": M_fu.mean(axis=(1, 2)).mean(), "hit": D_fu.mean(axis=(1, 2)).mean(), "iou": np.nan},
        "Centre guess (no model)": {"energy": ce.mean(), "hit": ch.mean(), "iou": cu.mean()}}

rng_b = np.random.default_rng(CONFIG["SEED"])
def boot_ci(d):
    d = d[~np.isnan(d)]
    m = [rng_b.choice(d, len(d)).mean() for _ in range(CONFIG["N_BOOT"])]
    return np.percentile(m, [2.5, 97.5])

base = per_img[per_img.variant == "Baseline"].set_index("img")
fold_means = per_fold.groupby(["variant", "fold"])[["energy", "hit", "iou"]].mean()
rows = []
for v in CONFIG["VARIANTS"]:
    d = per_img[per_img.variant == v].set_index("img")
    r = {"variant": v + (" (selected)" if v == SELECTED else "")}
    for m, name in [("energy", "energy inside ulcer"), ("hit", "pointing game"), ("iou", "IoU")]:
        r[f"{name} (%)"] = f"{100 * d[m].mean():.1f} ± {100 * fold_means.loc[v][m].std():.1f}"
    r["predicted as ulcer (%)"] = round(100 * d.pred_ulcer.mean(), 1)
    if v != "Baseline":
        diff = (d.energy - base.energy.loc[d.index]).values
        lo, hi = boot_ci(diff)
        r["energy diff vs Baseline (points)"] = f"{100 * np.nanmean(diff):+.2f} [{100 * lo:+.2f}, {100 * hi:+.2f}]"
        pv = wilcoxon(d.energy, base.energy.loc[d.index], nan_policy="omit").pvalue
        r["Wilcoxon p"] = "<0.0001" if pv < 1e-4 else f"{pv:.4f}"
        r["folds where better"] = f"{int((fold_means.loc[v].energy.values > fold_means.loc['Baseline'].energy.values).sum())}/5"
    rows.append(r)
for name, ref in refs.items():
    rows.append({"variant": name, "energy inside ulcer (%)": f"{100 * ref['energy']:.1f}",
                 "pointing game (%)": f"{100 * ref['hit']:.1f}",
                 "IoU (%)": "" if np.isnan(ref["iou"]) else f"{100 * ref['iou']:.1f}"})
summary = pd.DataFrame(rows).set_index("variant").astype(object).fillna("")
summary.to_csv("localization_summary.csv")
print("± = standard deviation over the 5 fold models. [ ] = 95% bootstrap interval.")
summary

## 7. Charts and example heatmaps
The example images were **picked at random with a fixed seed before the results were seen**, so they are not cherry-picked.

In [ ]:
V = CONFIG["VARIANTS"]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
means = [100 * per_img[per_img.variant == v].energy.mean() for v in V]
cis = [100 * boot_ci(per_img[per_img.variant == v].energy.values) for v in V]
cols = ["#888888" if v == "Baseline" else "#b22222" if v == SELECTED else "#1f3b63" for v in V]
ax[0].bar(V, means, yerr=np.abs(np.array(cis).T - means), color=cols, capsize=4)
ax[0].axhline(100 * refs["Chance (flat heatmap)"]["energy"], ls=":", c="k", label="chance (flat heatmap)")
ax[0].axhline(100 * refs["Centre guess (no model)"]["energy"], ls="--", c="k", label="centre guess (no model)")
ax[0].set_ylabel("Heatmap energy inside the ulcer (%)"); ax[0].legend(fontsize=8)
ax[0].set_title("Localisation on FUSeg (grey = baseline, red = selected)")
sel = per_img[per_img.variant == SELECTED].set_index("img").energy
ax[1].scatter(100 * base.energy, 100 * sel.loc[base.index], s=8, alpha=0.4, c="#1f3b63")
ax[1].plot([0, 100], [0, 100], c="k", lw=1)
ax[1].set_xlabel("Baseline: energy inside ulcer (%)"); ax[1].set_ylabel(f"{SELECTED}: energy inside ulcer (%)")
ax[1].set_title("Each dot is one image (above the line = attention better)")
plt.tight_layout(); plt.savefig("localization_chart.png", dpi=130); plt.show()

def overlay(a, img, cam, title):
    a.imshow(img); a.imshow(cam, cmap="jet", alpha=0.45, vmin=0, vmax=1); a.set_title(title, fontsize=8); a.axis("off")

fig, ax = plt.subplots(len(EXAMPLES), 3, figsize=(9, 3 * len(EXAMPLES)))
for r, i in enumerate(EXAMPLES):
    ax[r, 0].imshow(X_fu[i]); ax[r, 0].contour(M_fu[i], levels=[0.5], colors="lime", linewidths=1.5)
    ax[r, 0].set_title(f"FUSeg {meta.split[i]}/{meta.file[i]} (expert outline)", fontsize=8); ax[r, 0].axis("off")
    for c, v in enumerate(["Baseline", SELECTED], start=1):
        e, *_ = scores(ex_cam[v][r][None], M_fu[i][None], D_fu[i][None])
        overlay(ax[r, c], X_fu[i], ex_cam[v][r], f"{v}: energy in ulcer {100 * e[0]:.0f}%")
        ax[r, c].contour(M_fu[i], levels=[0.5], colors="white", linewidths=1)
plt.tight_layout(); plt.savefig("gradcam_fuseg_examples.png", dpi=110); plt.show()

fig, ax = plt.subplots(len(KD_EX), 3, figsize=(9, 3 * len(KD_EX)))
for r, i in enumerate(KD_EX):
    ax[r, 0].imshow(X_kd[i]); ax[r, 0].set_title(f"Kaggle DFU {kd.file[i]} (test fold {kd.fold[i]})", fontsize=8); ax[r, 0].axis("off")
    for c, v in enumerate(["Baseline", SELECTED], start=1):
        overlay(ax[r, c], X_kd[i], kd_cam[v][r], v)
plt.tight_layout(); plt.savefig("gradcam_kdfu_examples.png", dpi=110); plt.show()

## 8. Key numbers (printed from this run)

In [ ]:
d_sel = per_img[per_img.variant == SELECTED].set_index("img")
diff = (d_sel.energy - base.energy.loc[d_sel.index]).values
p = wilcoxon(d_sel.energy, base.energy.loc[d_sel.index], nan_policy="omit").pvalue
lo, hi = boot_ci(diff)
print(f"Images scored: {len(X_fu)} FUSeg crops, 5 fold models per variant")
print(f"Chance (flat heatmap):   {100 * refs['Chance (flat heatmap)']['energy']:.1f}% energy inside ulcer")
print(f"Centre guess (no model): {100 * refs['Centre guess (no model)']['energy']:.1f}%")
print(f"Baseline:                {100 * base.energy.mean():.1f}%")
print(f"{SELECTED} (selected):   {100 * d_sel.energy.mean():.1f}%")
print(f"Difference: {100 * np.nanmean(diff):+.2f} points, 95% CI [{100 * lo:+.2f}, {100 * hi:+.2f}], Wilcoxon p = {'<0.0001' if p < 1e-4 else f'{p:.4f}'}")
if p < 0.05 and np.nanmean(diff) > 0:
    print("-> Attention places significantly MORE of the heatmap on the ulcer than the baseline.")
elif p < 0.05:
    print("-> Attention places significantly LESS of the heatmap on the ulcer than the baseline.")
else:
    print("-> No significant difference in localisation between attention and the baseline.")
beats = base.energy.mean() > refs["Centre guess (no model)"]["energy"]
print("Baseline beats the centre guess:", "YES" if beats else "NO - the crops may be too centred to judge localisation")
print("Share of FUSeg crops predicted as ulcer (all of them are ulcers):",
      ", ".join(f"{v} {100 * per_img[per_img.variant == v].pred_ulcer.mean():.0f}%" for v in V))

## How to read the results
- **Section 4 / the loop lines:** the probability difference vs notebook 06 should be tiny (e.g. 1e-5). If the assert fails, the wrong inputs are attached.
- **Section 6:** the main table. Compare the selected variant with the Baseline. *Wilcoxon p < 0.05* means the difference is unlikely to be chance.
  Every model should also be clearly above both reference lines; otherwise the heatmaps are not finding the ulcer at all.
- **Section 7:** pictures for the thesis and slides. The green/white line is the expert ulcer outline.
- **"Predicted as ulcer":** every FUSeg image contains an ulcer, so this is a rough external check. FUSeg comes from a different hospital and camera, so lower numbers are expected.

**Limitations to state in the thesis**
- The crops are built from the masks. The random shift and the centre-guess line control for this, but the crops are still ulcer-centred close-ups.
- FUSeg is a different source than the training data (domain shift).
- Grad-CAM works on a 7×7 grid, so heatmaps are coarse. It shows *where the evidence comes from*, not *why*.

**Outputs:** `fuseg_scores_per_fold.csv`, `fuseg_scores_per_image.csv`, `localization_summary.csv`, `localization_chart.png`,
`gradcam_fuseg_examples.png`, `gradcam_kdfu_examples.png`.